# 01 — Bronze: Ingestão NYC TLC → S3

**Responsabilidade:** Armazenar os arquivos originais no S3 e registrar no Unity Catalog como tabela Delta preservando dado 100% original.

> Executar `00_config` antes deste notebook.

**Nota sobre download:** O Databricks Serverless não tem acesso à internet externa.
O download dos arquivos é feito localmente via `src/ingestion/download_to_s3.py`.

## Célula 1 — Carregar configurações

In [ ]:
%run "./00_config"

## Célula 2 — Verificar arquivos no S3

In [ ]:
# Verifica se os arquivos foram carregados corretamente
print("=== Arquivos Bronze no S3 ===")
try:
    files = dbutils.fs.ls(f"s3://{S3_BUCKET}/bronze/nyc_taxi/yellow/")
    for f in files:
        print(f"  {f.name}")
    print(f"\nTotal: {len(files)} pasta(s)")
except Exception as e:
    print(f"Nenhum arquivo encontrado: {e}")
    print("Execute src/ingestion/download_to_s3.py localmente primeiro")

## Célula 3 — Criar schema Bronze e registrar tabela Delta

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze')
print(f"Schema: {CATALOG}.bronze")

spark.sql(f'DROP TABLE IF EXISTS {BRONZE_TABLE}')

# Le cada mes separadamente — schemas diferentes entre arquivos
# unionByName: alinha por nome de coluna, NULL onde nao existe
# Preserva TODAS as colunas originais sem nenhuma transformacao

dfs = []
for month in MONTHS:
    path = f"s3://{S3_BUCKET}/bronze/nyc_taxi/yellow/year={YEAR}/month={month}/"
    print(f"Lendo mes {month}...")
    df = spark.read.parquet(path) \
           .withColumn('year',  F.lit(YEAR)) \
           .withColumn('month', F.lit(month))
    dfs.append(df)
    print(f"  OK: {df.count():,} registros | {len(df.columns)} colunas")

df_bronze = dfs[0]
for df in dfs[1:]:
    df_bronze = df_bronze.unionByName(df, allowMissingColumns=True)

print(f"\nTotal Bronze : {df_bronze.count():,} registros")
print(f"Total colunas: {len(df_bronze.columns)}")

(
    df_bronze.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(BRONZE_TABLE)
)
print(f"\nTabela registrada: {BRONZE_TABLE}")
print("  Tipo: Delta — dado original completo, sem transformacoes")

## Célula 4 — Validação Bronze

In [ ]:
spark.sql(f"DESCRIBE TABLE {BRONZE_TABLE}").show(30, truncate=False)
spark.sql(f'''
    SELECT year, month, COUNT(*) AS total_registros
    FROM {BRONZE_TABLE}
    GROUP BY year, month ORDER BY year, CAST(month AS INT)
''').show()
spark.sql(f'''
    SELECT vendor_id, passenger_count, total_amount,
           tpep_pickup_datetime, tpep_dropoff_datetime
    FROM {BRONZE_TABLE} LIMIT 5
''').show(truncate=False)